# Bad-channel check

Load every subject through `analysis/trf_pipeline` and report the bad channels recorded in the resulting MNE `Raw` object. 

In [1]:
from pathlib import Path

import pandas as pd
from eelbrain import load_pipeline
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis" / "trf_pipeline").is_dir():
    PROJECT_ROOT = Path.cwd().parents[1]

e = load_pipeline(PROJECT_ROOT / "analysis" / "trf_pipeline")
subjects = sorted(e.get_field_values("subject"), key=int)
print(f"Found {len(subjects)} subjects: {subjects[0]}-{subjects[-1]}")

INFO    :  *** AliceComprehension initialized with root /Users/yanyuwoo/Data/bids on 2026-08-11 19:05:48 ***
INFO    :  Using eelbrain 0.43.0a1, mne 1.11.0.
Found 49 subjects: 01-49


In [2]:
def check_subject(subject):
    try:
        raw = e.load_raw(subject=subject, raw="raw", preload=False)
    except Exception as error:
        return {
            "subject": subject,
            "status": "error",
            "n_channels": pd.NA,
            "channel_29_present": pd.NA,
            "channel_25_present": pd.NA,
            "n_bad_channels": pd.NA,
            "bad_channels": pd.NA,
            "error": f"{type(error).__name__}: {error}",
        }

    bad_channels = list(raw.info["bads"])
    return {
        "subject": subject,
        "status": "ok",
        "n_channels": len(raw.ch_names),
        "channel_29_present": "29" in raw.ch_names,
        "channel_25_present": "25" in raw.ch_names,
        "n_bad_channels": len(bad_channels),
        "bad_channels": ", ".join(bad_channels),
        "error": "",
    }


bad_channel_check = pd.DataFrame(check_subject(subject) for subject in subjects)
display(bad_channel_check)

,subject,status,n_channels,channel_29_present,channel_25_present,n_bad_channels,bad_channels,error
0,01,ok,62,False,True,0,,
1,02,ok,62,False,True,0,,
2,03,ok,62,False,True,0,,
3,04,ok,62,False,True,0,,
4,05,ok,61,False,True,0,,
5,06,ok,62,False,True,0,,
6,07,ok,62,False,True,0,,
7,08,ok,62,False,True,0,,
8,09,ok,62,False,True,0,,
9,10,ok,62,False,True,0,,


In [3]:
successful = bad_channel_check[bad_channel_check["status"] == "ok"]
with_bad_channels = successful[successful["n_bad_channels"] > 0]
failed = bad_channel_check[bad_channel_check["status"] == "error"]

print(f"Successfully checked: {len(successful)}/{len(bad_channel_check)}")
print(f"Subjects with bad channels: {len(with_bad_channels)}")
print(f"Subjects that failed to load: {len(failed)}")

if not with_bad_channels.empty:
    display(with_bad_channels[["subject", "n_bad_channels", "bad_channels"]])
if not failed.empty:
    display(failed[["subject", "error"]])

Successfully checked: 49/49
Subjects with bad channels: 0
Subjects that failed to load: 0
